<a href="https://colab.research.google.com/github/pink3y3/link_prediction-citation_network/blob/main/gae_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GAE (Graph AutoEncoder)
goal : given 2 papers, predict whether a citation link exists
A GAE learns a vector(node embedding) for every paper

The vectors should contain information about:
paper's features
it's neighbours
it's position in the graph

Node Features (1433)
        ↓
GCN Layer
        ↓
GCN Layer
        ↓
Embeddings Z
        ↓
Dot Product
        ↓
Sigmoid
        ↓
Link Probability

In [ ]:
!pip install torch-geometric
from torch_geometric.datasets import Planetoid

dataset=Planetoid(root='/data/Cora',name='Cora')
data=dataset[0]


In [ ]:
print(data)
print(data.x.shape) #2708 papers, 1433 features per paper
print(data.y.shape) #one label per paper
print(data.edge_index.shape) #10556 edges

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
torch.Size([2708, 1433])
torch.Size([2708])
torch.Size([2, 10556])


In [ ]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True
)

train_data, val_data, test_data = transform(data)

"""
edge_label - (1 = edge exists , 0 = edge doesn't exist)
edge_label_index - which node pairs we're testing
"""


"\nedge_label - (1 = edge exists , 0 = edge doesn't exist)\nedge_label_index - which node pairs we're testing\n"

In [ ]:
import torch
from torch_geometric.nn import GCNConv

# Import PyTorch and the Graph Convolutional Network (GCN) layer from PyTorch Geometric.


class Encoder(torch.nn.Module):
    # Define a neural network class called Encoder.
    # Every neural network in PyTorch inherits from torch.nn.Module.

    def __init__(self, in_channels, out_channels):
        # Constructor method.
        # in_channels = number of input features per node.
        # out_channels = size of the final embedding produced for each node.

        super().__init__()
        # Initialize the parent torch.nn.Module class.

        self.conv1 = GCNConv(in_channels, 128)
        # First graph convolution layer.
        # Input: in_channels features per node.
        # Output: 128 features per node.
        #
        # Example for Cora:
        # (2708 nodes, 1433 features)
        #           ↓
        # (2708 nodes, 128 features)

        self.conv2 = GCNConv(128, out_channels)
        # Second graph convolution layer.
        # Input: 128 features per node.
        # Output: out_channels features per node.
        #
        # Example:
        # (2708, 128)
        #      ↓
        # (2708, 64)


    def forward(self, x, edge_index):
        # Defines how data flows through the network.
        #
        # x = node feature matrix
        #     Shape: (number_of_nodes, number_of_features)
        #
        # edge_index = graph connectivity information
        #     Stores which nodes are connected.

        x = self.conv1(x, edge_index)
        # Apply the first graph convolution.
        #
        # For every node:
        # 1. Gather information from neighboring nodes.
        # 2. Combine neighbor features with the node's own features.
        # 3. Apply learnable weights.
        #
        # Output shape:
        # (num_nodes, 128)

        x = torch.relu(x)
        # Apply ReLU activation function.
        #
        # ReLU(x) = max(0, x)
        #
        # Example:
        # [-3, 2, -1, 5]
        #      ↓
        # [0, 2, 0, 5]
        #
        # Adds non-linearity so the network can learn complex patterns.

        x = self.conv2(x, edge_index)
        # Apply the second graph convolution.
        #
        # Again aggregates information from neighbors,
        # but now using the 128-dimensional representations
        # learned by the first layer.
        #
        # Output shape:
        # (num_nodes, out_channels)

        return x
        # Return the final node embeddings.
        #
        # Example:
        # (2708, 64)
        #
        # Each node now has a 64-dimensional embedding vector.
        #
        # These embeddings contain information about:
        # - the node's original features
        # - its neighboring nodes
        # - the graph structure
        #
        # The Graph Autoencoder (GAE) decoder will later use
        # these embeddings to predict whether two nodes
        # should have an edge between them.